# 🍽️ S3C Smart Canteen: Food Waste Training (Cloud GPU)

Notebook ini dibuat untuk melatih model YOLO menggunakan GPU gratis di Google Colab. Ini jauh lebih cepat daripada training di laptop.

### Langkah-langkah:
1. Buka Menu **Runtime** > **Change runtime type** > Pilih **T4 GPU**.
2. Jalankan cell di bawah secara berurutan.

## 1. Install Dependencies

In [1]:
!pip install --upgrade ultralytics roboflow
import ultralytics
from ultralytics import YOLO
from roboflow import Roboflow
import os
from google.colab import files

# Verification
print(f"Ultralytics version: {ultralytics.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 103.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings w

## 2. Download Dataset dari Roboflow
Masukkan API Key Roboflow Anda di bawah.
*(Cara ambil API Key: Buka Roboflow -> Settings -> Workspace Settings -> Copy Private API Key)*

In [4]:
rf = Roboflow(api_key="8SEqM5jS0qJ4PfUjzQ1h")
project = rf.workspace("fauziniko").project("menu-mbg-mbkm-ylkbs")
version = project.version(3)
dataset = version.download("yolov11")

# Update data.yaml agar path-nya sesuai di Colab
import yaml
with open(f"{dataset.location}/data.yaml", 'r') as f:
    data = yaml.safe_load(f)

data['path'] = dataset.location
with open(f"{dataset.location}/data.yaml", 'w') as f:
    yaml.dump(data, f)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to menu-mbg-mbkm-3 in yolov11:: 100%|██████████| 2213/2213 [00:00<00:00, 3172.68it/s]


## 3. Mulai Training (GPU)
Proses ini biasanya memakan waktu 15-30 menit.

In [5]:
from ultralytics import YOLO
import yaml
import os
import shutil
from sklearn.model_selection import train_test_split

base_path = os.path.abspath(dataset.location)
subdirs = os.listdir(base_path)

# 1. Ensure 'valid' folder exists. If not, split from 'train'.
if 'valid' not in subdirs and 'val' not in subdirs and 'train' in subdirs:
    print("Validation folder missing. Creating a split from train...")
    train_img_path = os.path.join(base_path, 'train', 'images')
    val_img_path = os.path.join(base_path, 'valid', 'images')
    train_lbl_path = os.path.join(base_path, 'train', 'labels')
    val_lbl_path = os.path.join(base_path, 'valid', 'labels')

    os.makedirs(val_img_path, exist_ok=True)
    os.makedirs(val_lbl_path, exist_ok=True)

    images = [f for f in os.listdir(train_img_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
    train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

    for img in val_imgs:
        shutil.move(os.path.join(train_img_path, img), os.path.join(val_img_path, img))
        label = os.path.splitext(img)[0] + '.txt'
        if os.path.exists(os.path.join(train_lbl_path, label)):
            shutil.move(os.path.join(train_lbl_path, label), os.path.join(val_lbl_path, label))
    print(f"Moved {len(val_imgs)} images to validation set.")

# 2. Update data.yaml with verified paths
with open(f"{dataset.location}/data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = base_path
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'
data_config['test'] = 'test/images' if 'test' in os.listdir(base_path) else 'valid/images'

with open(f"{dataset.location}/data.yaml", 'w') as f:
    yaml.dump(data_config, f)

# 3. Start Training
model = YOLO('yolo11n.pt')
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    plots=True,
    device=0
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
from google.colab import drive
import os
import shutil

# 1. Menyambungkan (Mount) Google Drive
drive.mount('/content/drive')

# 2. Menentukan lokasi file asli dan lokasi tujuan di Drive
# Catatan: Berdasarkan log Anda, folder penyimpanannya adalah 'train-4'
source_path = '/content/runs/detect/train-4/weights/best.pt'

# Buat folder khusus di Google Drive Anda (bisa diganti namanya)
drive_folder = '/content/drive/MyDrive/S3C_Models'
os.makedirs(drive_folder, exist_ok=True)

destination_path = os.path.join(drive_folder, 'best_food_waste.pt')

# 3. Proses penyalinan
if os.path.exists(source_path):
    shutil.copy(source_path, destination_path)
    print(f"✅ SUKSES! Model berhasil diamankan ke Google Drive di folder: {destination_path}")
else:
    print("❌ File best.pt tidak ditemukan. Cek apakah folder 'train-4' benar atau training belum selesai.")

## 4. Download Model Hasil Training
Jalankan cell ini untuk mendownload file `best.pt`.
Setelah terdownload, pindahkan file tersebut ke folder `models/food_seg_model.pt` di laptop Anda.

In [ ]:
best_model_path = 'runs/detect/train/weights/best.pt'
if os.path.exists(best_model_path):
    files.download(best_model_path)
    print("Model 'best.pt' sedang didownload...")
else:
    # Jika nama folder berbeda (misal train2, train3)
    print("Mencari file model di folder lain...")
    !find runs/ -name "best.pt"